## <b><font color='darkblue'>Preface</font></b>
<font size='3ptx'><b>This notebook is for this issue [[ResearchSpike] Study 臺灣證券交易所 OpenAPI #6](https://github.com/johnklee/finance_agent/issues/6)</b>. 目的是可以給定 Stock ID, 然後可以轉成 Stock Symbol 餵到 Yahoo Finance API 中. e.g. `2330` -> `2330.TW`</font>

這邊有 Survey 到幾種作法, 會在這個 Notebook 去評估 Pros & Cons.

### <b><font color='darkgreen'>Recommended Approaches</font></b>
- **Option 1 — Use TWSE / TPEX official listings (Best Practice)**

### <b><font color='darkgreen'>Importing Packages</font></b>
Let's import the necessary packages here:

In [22]:
import os
from pathlib import Path
import requests

import pandas as pd

## <b><font color='darkblue'>Option 1 — Use TWSE / TPEX official listings (Best Practice)</font></b>
Download:
- TWSE listed stocks
- TPEX OTC stocks

Then generate Yahoo symbols dynamically. Example mapping table:

| stock_id | market | yahoo_symbol | type |
|---|---|---|---| 
| 2330 | TWSE | 2330.TW |  上市 |
| 8069 | TPEX | 8069.TWO | 上櫃 |

This is the most robust approach for production systems.

### <b><font color='darkgreen'>Step 1 — Fetch TWSE list</font></b>
You can use official open data from:
- TWSE OpenAPI  
  https://openapi.twse.com.tw/

- TPEx OpenAPI  
  https://www.tpex.org.tw/openapi/

### <b><font color='darkgreen'>Step 2 — Build Mapping</font></b>
We could utilize below code snippet to pull the databack into a local CSV file for reference.

In [8]:
os.getcwd()

'/mnt/sdb/Gitrepos/finance_agent/labs'

In [19]:
import io


TWSE_ISIN_URL = "https://isin.twse.com.tw/isin/C_public.jsp?strMode=2"


def _load_twse_table() -> pd.DataFrame:
    response = requests.get(
        TWSE_ISIN_URL,
        timeout=30,
    )
    response.raise_for_status()

    response.encoding = response.apparent_encoding or "cp950"

    tables = pd.read_html(io.StringIO(response.text))

    if not tables:
        raise ValueError("No HTML tables found.")

    return tables[0]

In [20]:
df = _load_twse_table()

In [21]:
df

,0,1,2,3,4,5,6
0,有價證券代號及名稱,國際證券辨識號碼(ISIN Code),上市日,市場別,產業別,CFICode,備註
1,股票,股票,股票,股票,股票,股票,股票
2,1101 台泥,TW0001101004,1962/02/09,上市,水泥工業,ESVUFR,NaN
3,1102 亞泥,TW0001102002,1962/06/08,上市,水泥工業,ESVUFR,NaN
4,1103 嘉泥,TW0001103000,1969/11/14,上市,水泥工業,ESVUFR,NaN
...,...,...,...,...,...,...,...
31997,01002T 土銀國泰R1,TW00001002T6,2005/10/03,上市,NaN,CBCIXU,NaN
31998,01004T 土銀富邦R2,TW00001004T2,2006/04/13,上市,NaN,CBCIXU,NaN
31999,01007T 兆豐國泰R2,TW00001007T5,2006/10/13,上市,NaN,CBCIXU,NaN
32000,01009T 王道圓滿R1,TW00001009T1,2018/06/21,上市,NaN,CBCIXU,NaN


In [12]:
# constant
TWSE_ISIN_URL = "https://isin.twse.com.tw/isin/C_public.jsp?strMode=2"

TWSE_CACHED_CSV_PATH = Path(os.getcwd()) / "twse_listed.csv"

In [26]:
def build_twse_symbol_mapping(
    local_csv_path: str | Path | None = TWSE_CACHED_CSV_PATH,
    refresh: bool = False,
) -> dict[int, str]:
    """Build a mapping from Taiwan stock ID to Yahoo Finance symbol.

    The mapping is generated from the Taiwan Stock Exchange (TWSE)
    securities listing.

    If a local cache file is provided, the function will load the
    previously downloaded data from disk. Otherwise, it retrieves the
    latest listing from TWSE.

    Example:
        >>> mapping = build_twse_symbol_mapping()
        >>> mapping[2330]
        '2330.TW'

    Args:
        local_csv_path:
            Optional path to a cached CSV file.

        refresh:
            If True, always download the latest data from TWSE and
            overwrite the cache.

    Returns:
        A dictionary mapping stock IDs to Yahoo Finance symbols.

    Notes:
        - Only TWSE-listed securities are included.
        - OTC securities (Yahoo suffix ".TWO") are not included.
        - The TWSE source format may change over time.
    """
    csv_path = Path(local_csv_path) if local_csv_path is not None else None

    if csv_path is not None and csv_path.exists() and not refresh:
        df = pd.read_csv(csv_path, encoding="utf-8-sig")
    else:
        df = _download_twse_listing()

        if csv_path is not None:
            csv_path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            df.to_csv(
                csv_path,
                index=False,
                encoding="utf-8-sig",
            )

    return _build_mapping_from_dataframe(df)


def _download_twse_listing() -> pd.DataFrame:
    """Download and parse the TWSE securities listing."""
    response = requests.get(
        TWSE_ISIN_URL,
        timeout=30,
    )
    response.raise_for_status()

    # TWSE may serve Big5/CP950 content while not providing
    # a reliable charset declaration.
    response.encoding = response.apparent_encoding or "cp950"

    tables = pd.read_html(io.StringIO(response.text))

    if not tables:
        raise ValueError("No tables found in TWSE listing page.")

    return tables[0]


def _build_mapping_from_dataframe(
    df: pd.DataFrame,
) -> dict[int, str]:
    """Convert a TWSE listing DataFrame to a Yahoo symbol mapping."""
    mapping: dict[int, str] = {}

    for value in df.iloc[:, 0].dropna():
        parts = str(value).strip().split()

        if not parts:
            continue

        stock_id = parts[0]

        if stock_id.isdigit():
            mapping[int(stock_id)] = f"{stock_id}.TW"

    return mapping

In [27]:
%%time
twse_mapping = build_twse_symbol_mapping()

CPU times: user 3.58 s, sys: 236 ms, total: 3.82 s
Wall time: 5.28 s


In [28]:
%%time
twse_mapping = build_twse_symbol_mapping()

CPU times: user 147 ms, sys: 7.97 ms, total: 155 ms
Wall time: 151 ms


In [30]:
twse_mapping[2330]

'2330.TW'

## <b><font color='darkblue'>Option 2 - Easier Alternative In Using `twstock` package
The Python package twstock already maintains Taiwan stock metadata.

GitHub:
https://github.com/mlouielu/twstock

Install:
```shell
$ pip install twstock
```

### <b><font color='darkgreen'>Example</font></b>

In [31]:
import twstock

stock_id = "2330"

stock = twstock.codes[stock_id]

print(stock.name)
print(stock.market)

台積電
上市


In [32]:
def yahoo_symbol(stock_id: str) -> str:
    stock = twstock.codes[stock_id]

    if stock.market == "上市":
        return f"{stock_id}.TW"
    else:
        return f"{stock_id}.TWO"


print(yahoo_symbol("2330"))

2330.TW


## <b><font color='darkblue'>Option 3 - Another Good Option — Use `FinMind`</font></b>
Official site:
https://finmindtrade.com/

Provides:
- stock metadata
- market type
- historical prices
- Taiwan finance datasets

You can query stock info directly.

### <b><font color='darkgreen'>Store a `stock_metadata` table</font></b>
Example schema:
```sql
CREATE TABLE stock_metadata (
    stock_id VARCHAR(10) PRIMARY KEY,
    company_name TEXT,
    market VARCHAR(10),
    yahoo_symbol VARCHAR(20),
    updated_at TIMESTAMP
);
```

Then refresh daily from:
- TWSE OpenAPI
- TPEX OpenAPI

Advantages:
- no hardcoding
- fast lookup
- supports delisted stocks
- easier joins with historical data
- future extensibility

## <b><font color='darkblue'>Practical Recommendation</font></b>
For your use case:

| Approach | Recommendation |
|---|---|
| Small script | `twstock` |
| Production system | Local metadata table |
| Enterprise-grade | Official exchange APIs + DB cache |

`twstock` is excellent for prototyping, but for long-term stability, keeping your own metadata cache is better.